# <center>[?? ?????????????? ??????? ? ????????](https://stepik.org/lesson/872530/)</center>

### ?????????? ????????

<img src='https://raw.githubusercontent.com/a-milenkin/Competitive_Data_Science/main/images/pipelines.jpg'/>
<br>

<p><font size="3" face="Arial" font-size="large"><ul type="square">
    
<li><a href="#c1">?? ??? ?????? ??? ????????? </a></li>
<li><a href="#look1">?? ?????????? ????????</a>
<li><a href="#check1"> ???? ?????? ????????</a>
<li><a href="#6">?? ?????? ? ??????????</a>

</li></ul></font></p>


<div class="alert alert-info">

* ??? ?? ??? ????? ????????, ??? ?????? ??????? ?? `????????`, ??? ?????? ? ??? ???????????? ???, ? ???-?? ?????????? ? ??? ?????? ?? ??????????.
* ?? ???? ??????????? ???????????, ??????? ????????? ??????? ??? ????????? ? ???? ????? ??????????. ?? ??? ? ??????? ?????? ????? ????!
* ?? ??? ??????????? ? ???????????, ???.
    

<div class="alert alert-info">
    
??????? ?? ????????? ??? `sklearn.Pipelines` - ?????? ????????? ??? ??????? ???????? ? ????????? ?? `Feature Engineering`? ?? ???????? 10 ??????? ? ???? ????????.

## ??????????? ??????????

In [ ]:
!pip install catboost -q

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import lightgbm as lgbm
import xgboost as xgb
import catboost as cb

In [ ]:
# pip install xgboost -U -q

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## ????????? ??????

In [ ]:
from sklearn import preprocessing
data_root = "https://raw.githubusercontent.com/a-milenkin/Competitive_Data_Science/main/data/"
data = pd.read_csv(data_root + 'quickstart_train.csv')

categorical_features = ['model', 'car_type', 'fuel_type']

for cat in categorical_features:
    lbl = preprocessing.LabelEncoder()
    data[cat] = lbl.fit_transform(data[cat].astype(str))
    data[cat] = data[cat].astype('category')

data.info()

### ???????? ??????? ?? ????????????? ? ?????????

In [ ]:
# ???????? ??????? ?????????? ?????? ???????
class_names = np.unique(data['target_class'])
data['target_class'] = data['target_class'].replace(class_names, np.arange(data['target_class'].nunique()))

In [ ]:
cols2drop = ['car_id', 'target_reg', 'target_class']
categorical_features = ['model', 'car_type', 'fuel_type']
numerical_features = [c for c in data.columns if c not in categorical_features and c not in cols2drop]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(data.drop(cols2drop, axis=1),
                                                    data['target_class'],
                                                    test_size=.25,
                                                    stratify=data['target_class'],
                                                    random_state=42)
print(X_train.shape, X_val.shape)

# <center> ?? ??????? 3 ??????

#  ???? ?????? `CatBoost`

In [ ]:
params_cat = {
             'n_estimators' : 700,
              # 'learning_rate': .03,
              'depth' : 3,
              'verbose': False,
              'use_best_model': True,
              'cat_features' : categorical_features,
              'text_features': [],
              # 'train_dir' : '/home/jovyan/work/catboost',
              'border_count' : 64,
              'l2_leaf_reg' : 1,
              'bagging_temperature' : 2,
              'rsm' : 0.51,
              'loss_function': 'MultiClass',
              'auto_class_weights' : 'Balanced', #try not balanced
              'random_state': 42,
              'use_best_model': False,
              # 'custom_metric' : ['AUC', 'MAP'] # ?? ???????? ?????? Sklearn.Pipelines
         }

In [ ]:
cat_model = cb.CatBoostClassifier(**params_cat)

# ???? ?????? `LightGBM`

In [ ]:
categorical_features_index = [i for i in range(data.shape[1]) if data.columns[i] in categorical_features]
params_lgbm = {
    "num_leaves": 200,
    "n_estimators": 1500,
    # "max_depth": 7,
    "min_child_samples": None,
    "learning_rate": 0.001,
    "min_data_in_leaf": 5,
    "feature_fraction": 0.98,
    # "categorical_feature": cat_cols,
    'reg_alpha' : 3.0,
    'reg_lambda' : 5.0,
    'categorical_feature': categorical_features_index
}

In [ ]:
lgbm_model = lgbm.LGBMClassifier(**params_lgbm)

# ???? ?????? `XGBoost`

In [ ]:
params_xgb = {
    "eta": 0.05,
    'n_estimators' : 1500,
    "max_depth": 6,
    "subsample": 0.7,
    # "colsample_bytree": 0.95,
    'min_child_weight' : 0.1,
    'gamma': .01,
    'reg_lambda' : 0.1,
    'reg_alpha' : 0.5,
    "objective": "reg:linear",
    "eval_metric": "mae",
    'tree_method' : 'hist', # Supported tree methods for cat fs are `gpu_hist`, `approx`, and `hist`.
    'enable_categorical' : True

}

In [ ]:
xgb_model = xgb.XGBClassifier(**params_xgb)

# <center> ?? ???????? ???????

In [ ]:
!pip3 install -U scikit-learn -q

In [ ]:
# ??????????????? ????? ??????????? ??? ?????????
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector

In [ ]:
# ??????????????? ???????? ??? ?????????? ?????????
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder, MinMaxScaler

In [ ]:
# ????????? ?????? ??? ?????????? ????????
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

In [ ]:
# ??????? ????????????
import sklearn
sklearn.set_config(display='diagram')

from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)

### ?????????????? ??????
??? ?????? ??? ?????? ??????? ???? ???????????

In [ ]:
# ???????? ???????? ????? ?????? ????????? ? ?????? ohe
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))])

In [ ]:
# ???????? ???????? ??????? ????????? ? ?????? ????????????
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler())
])

In [ ]:
# ???????? ??? ?????????? ???????????? ? ????
preprocessor = ColumnTransformer(transformers=[
    ("numerical", numerical_transformer, numerical_features),
    ("categorical", categorical_transformer, categorical_features)])

preprocessor

In [ ]:
preprocessor.transformers[0]

# <center> ???? ?????? ????????

In [ ]:
# ?????? ??????? ???????
estimators = [


    ("ExtraTrees",  make_pipeline(preprocessor, ExtraTreesClassifier(n_estimators = 10_000, max_depth = 6, min_samples_leaf = 2,
                                                              bootstrap = True, class_weight = 'balanced', # ccp_alpha = 0.001,
                                                              random_state = 75, verbose=False, n_jobs=-1,))),


    ("XGBoost", xgb_model),
    ("LightGBM", lgbm_model),
    ("CatBoost", cat_model),

    # ??, ??? ?? ???? ???????? ? ????????
    # ("SVM", make_pipeline(preprocessor, LinearSVC(verbose=False))),
    # ("MLP", make_pipeline(preprocessor, MLPClassifier(verbose=False, hidden_layer_sizes=(100, 30, ), alpha=0.001,random_state=75, max_iter = 1300, ))),
    ("Random_forest",  make_pipeline(preprocessor, RandomForestClassifier(n_estimators = 15_000, max_depth = 7,
                                                              min_samples_leaf = 2,
                                                              warm_start = True, n_jobs=-1,
                                                              random_state = 75, verbose=False))),



]

# ? ???????? ????-?????? ????? ???????????? LogisticRegression
meta_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(verbose=False),
    # final_estimator=RandomForestClassifier(n_estimators = 10_000,
                                           # max_depth = 5,
                                           # verbose=False),
    n_jobs=-1,
    verbose=False,
)

stacking_classifier = meta_model
stacking_classifier

In [ ]:
stacking_classifier.fit(X_train, y_train)

In [ ]:
# for i in stacking_classifier.estimators:
#     print(i[0])

In [ ]:
# dir(stacking_classifier)

In [ ]:
corr_df = pd.DataFrame()

for model, (name, _) in zip(stacking_classifier.estimators_, stacking_classifier.estimators):
    preprocessed = stacking_classifier.estimators[0][1].steps[0][1].fit(X_train, y_train).transform(X_val)
    print(name, 'accuracy: ', round(accuracy_score(model.predict(X_val), y_val), 4))

    corr_df[name] = model.predict(X_val)


In [ ]:
corr_df.corr().style.background_gradient(cmap="RdYlGn")

In [ ]:
# Random_forest ?????? ??????????? ? ??????? ????????,
# ??????? ?? ??????? ???????? ????????.
# ?????????? ??? ??????

In [ ]:
print('ensemble score:', round(accuracy_score(stacking_classifier.predict(X_val), y_val), 4))
# ??? ?? ?????? ???????? ???????? ???????? ??????? ??????? ???????? ???????

# ???????????

* ?? ??, ???? ???????? ?????, ?? ???? ????? **"??"** ? ???? ??????????
* ?? ??? ? ???????? ????-?????? ?????????????? `LogisticRegression`, ??? ?? ???? ???????? ??????? ??????????, ?? ? ?????-??????????.
* ?? ?????? ??? ??????? ?????? ?????? ???????? ??????? ???? (???? ?????? `RandomForest` ???? ??????????)
* ?? ??????? ????? ?????????, ??????? ????-?????? ??? ???????? ??? ???? ????????? ????? ??????? meta-??????.
* ?? ????? ? ??????????? ?? ??????? ???????, ???? ??????, ????? ??? `RandomForestClassifier` ????? ????????? ??????? ???????????.
* ?? ? ?????? `pipeline` ? `Sklearn` ??? ??????? ???????. ???? ????? ???-?? ??????.

* ?? ???? ????? ????????? ? `pipeline`. ???????? `eval_set` ??? `early-stopping`? ??? ????? `train` ?? `LightGBM`
* ??? ???? ???? ?? ???? ?????

# <center> ?? ?????????????? ??????????
- [?????? ?????????? ????????? ??? ??????? ? ????????](https://alexanderdyakonov.wordpress.com/2017/03/10/c%D1%82%D0%B5%D0%BA%D0%B8%D0%BD%D0%B3-stacking-%D0%B8-%D0%B1%D0%BB%D0%B5%D0%BD%D0%B4%D0%B8%D0%BD%D0%B3-blending/)
- [?????? ???????, ???  `LigthGBM` ? ???????? ???? ??????](https://www.youtube.com/watch?v=aMlpeDOjib8)

# <center> ?? ?????? ? ??????????

<div class="alert alert-info">

* `sklearn.Pipelines` ??? ????? ??????? ??????????, ??????????? ????????? ???? ??????? ???????? ?????? ? ???? ????????
* ????? ????????? ????? ?????? ? ??????? ????? ????????? ?? ?????????.
* ??? ???? ?? ??? ????????????, ??????? ????? ???????????? ?? ?????? ?? ?????????????, ?? ? ? ??????? ?????? ?????? ????? ???????????? ? ????????.